# Shift-Scan: Evaluation & Ablation Notebook

This notebook evaluates saved Shift-Scan checkpoints using the project's `eval_lm.py` and summarizes reported results. It supports:
- benchmark evaluation (lm-evaluation-harness tasks)
- optional long-context loss-by-position evaluation on FineWeb-EDU
- comparison of multiple checkpoints by their JSON outputs

**Use a GPU runtime.** The provided `eval_lm.py` implementation requires CUDA. This is evaluation, not training. Start with a small benchmark `--limit` before a full evaluation.

> Important: use checkpoints produced by the project's own `train_qat.py` and keep a baseline/ablation's evaluation protocol identical.

## 1. Runtime and GPU check

In Colab: **Runtime → Change runtime type → T4 GPU** (or another available GPU).

In [ ]:
import torch, sys, os
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Select a GPU runtime: this project eval_lm.py requires CUDA.")

## 2. Upload the Shift-Scan project ZIP

Upload `shift_scan_advanced_project(1).zip` (or the source ZIP containing the `shift_scan/` directory) from your computer. If you keep it in Google Drive, you can instead mount Drive and set `ZIP_PATH` to its path.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select the Shift-Scan project ZIP
ZIP_PATH = next((name for name in uploaded if name.lower().endswith(".zip")), None)
assert ZIP_PATH, "Upload a .zip project archive"
print("Using:", ZIP_PATH)

In [ ]:
import zipfile, os
WORK_DIR = "/content/shift_scan_eval"
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(WORK_DIR)
# Find project root containing eval_lm.py and ssm_core.py
PROJECT_DIR = None
for root, dirs, files_ in os.walk(WORK_DIR):
    if "eval_lm.py" in files_ and "ssm_core.py" in files_:
        PROJECT_DIR = root
        break
assert PROJECT_DIR, "Could not locate eval_lm.py and ssm_core.py in uploaded ZIP"
print("Project root:", PROJECT_DIR)
print("eval_lm.py exists:", os.path.isfile(os.path.join(PROJECT_DIR, "eval_lm.py")))

!pip install -r "{PROJECT_DIR}/requirements.txt"
!pip install "lm-eval==0.4.13"

In [ ]:
%pip install -r "$PROJECT_DIR/requirements.txt"
%pip install "lm-eval==0.4.13" 

## 4. Select checkpoint

Put your checkpoint (`last.pt`) in Google Drive or upload it. A Kaggle output checkpoint is not automatically accessible from Colab: download it from Kaggle Output first, or use a shared Drive location.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# EDIT this to the real checkpoint path in your mounted Drive.
# Example: /content/drive/MyDrive/ShiftScan/last.pt
CKPT_PATH = "/content/drive/MyDrive/ShiftScan/last.pt"
assert os.path.isfile(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
print("Checkpoint size (GB):", round(os.path.getsize(CKPT_PATH)/1024**3, 3))

## 5. Quick evaluation (smoke test)

First run with `--limit 20` to verify checkpoint loading and evaluation. Set `TASKS` to tasks supported/installed by lm-eval. This project's default tasks are shown below.

In [ ]:
import subprocess
os.chdir(PROJECT_DIR)
RESULT_DIR = "/content/shift_scan_eval/results"
os.makedirs(RESULT_DIR, exist_ok=True)
TASKS = "lambada_openai,hellaswag,piqa,arc_easy,arc_challenge,winogrande,wikitext"
QUICK_JSON = os.path.join(RESULT_DIR, "quick_eval.json")
cmd = [
    sys.executable, "eval_lm.py",
    "--ckpt", CKPT_PATH,
    "--tasks", TASKS,
    "--batch-size", "2",
    "--limit", "20",
    "--dtype", "fp16",
    "--out", QUICK_JSON,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
print("Saved:", QUICK_JSON)

## 6. Full evaluation

After the smoke test succeeds, run a full task evaluation (omit `--limit`). The tasks can take a long time and consume GPU runtime. For comparability, use the same task list, dtype, and settings for each ablation.

In [ ]:
FULL_JSON = os.path.join(RESULT_DIR, "shift_scan_full_eval.json")
cmd = [
    sys.executable, "eval_lm.py",
    "--ckpt", CKPT_PATH,
    "--tasks", TASKS,
    "--batch-size", "2",
    "--dtype", "fp16",
    "--out", FULL_JSON,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
print("Saved:", FULL_JSON)

## 7. Optional: long-context loss by position

This evaluates loss by position buckets on held-out FineWeb-EDU text at lengths up to 8192 tokens. It downloads data, so internet access is needed. This project's implementation defaults to 8 sequences and lengths `[512, 1024, 2048, 4096, 8192]`; this may be memory/time intensive on a T4. Run it after the benchmark smoke test, and if OOM occurs, use the lower-memory adjustment described below.

In [ ]:
LONG_JSON = os.path.join(RESULT_DIR, "shift_scan_long_context.json")
cmd = [
    sys.executable, "eval_lm.py",
    "--ckpt", CKPT_PATH,
    "--tasks", "lambada_openai",
    "--limit", "20",
    "--ppl-length",
    "--batch-size", "1",
    "--dtype", "fp16",
    "--out", LONG_JSON,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
print("Saved:", LONG_JSON)

## 8. Read and plot result JSON

Use the same process to compare different checkpoints: evaluate each one to a distinct JSON filename, then load those files here.

In [ ]:
import json, pandas as pd
with open(FULL_JSON) as f:
    result = json.load(f)
rows = []
for task, metrics in result.get("results", {}).items():
    for metric, value in metrics.items():
        if isinstance(value, (int, float)) and "stderr" not in metric:
            rows.append({"task": task, "metric": metric, "value": value})
results_df = pd.DataFrame(rows)
display(results_df)

In [ ]:
# Optional export to CSV
CSV_PATH = os.path.join(RESULT_DIR, "benchmark_metrics.csv")
results_df.to_csv(CSV_PATH, index=False)
print("CSV:", CSV_PATH)
from google.colab import files
files.download(CSV_PATH)

## Research reporting cautions

- The first run is a smoke test; `--limit 20` is not a final benchmark.
- Report which checkpoint, task list, few-shot setting, data split, dtype, and limit were used.
- Compare baseline and ablations under identical settings.
- Report evaluation loss/perplexity and benchmark metrics separately; don't infer throughput or memory savings from quantization settings alone.
- The long-context loss-by-position test measures this implementation's behavior on the sampled FineWeb-EDU stream; it is not by itself proof of stable deployment at every context length.